<h1 style="text-align: center;">Phase 2 — Dataset V2.0: Feature Engineering and Model Experimentation</h1>



### Purpose

This notebook marks the second phase of experimentation in our football match outcome prediction project. Following the limited predictive performance achieved with Dataset V1.0 and V1.2, the objective is to investigate whether a more comprehensive dataset can improve both overall accuracy and balanced recall across **Home Win, Draw, and Away Win**.

### Experimental Approach

The initial strategy is to construct **Dataset V2.0** by integrating all relevant database tables and extracting meaningful features available before each match, while carefully preventing data leakage. The resulting dataset will be evaluated using selected high-performing machine learning algorithms, followed by hyperparameter tuning and systematic feature elimination to assess the contribution of different features.

The final results will be compared against previous experiments to determine whether richer pre-match information provides meaningful predictive improvements. Regardless of the outcome, the findings, limitations, and conclusions will be documented to support an evidence-based final assessment of the project.

----

## Dataset V2.0 — Data Integration and Preparation

This section begins the reconstruction of Dataset V2.0 using the original European Soccer Database. All seven relational tables are loaded and inspected before integration to ensure their structure and dimensions are understood.

The `Match` table will serve as the base dataset, while the remaining tables will be joined through their corresponding relationships. The objective is to preserve one record per match, incorporate relevant pre-match information, and subsequently identify data leakage, missing values, and unnecessary features.

In [1]:
import pandas as pd 
import duckdb
import sqlite3

In [2]:
connection = sqlite3.connect("../Data/database.sqlite")

In [3]:
match_df = pd.read_sql("SELECT * FROM Match", connection)
match_df.head()

,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,SJA,VCH,VCD,VCA,GBH,GBD,GBA,BSH,BSD,BSA
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.65,3.40,4.50,1.78,3.25,4.00,1.73,3.40,4.20
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.80,2.00,3.25,3.25,1.85,3.25,3.75,1.91,3.25,3.60
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.35,3.25,2.65,2.50,3.20,2.50,2.30,3.20,2.75
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,7.50,1.45,3.75,6.50,1.50,3.75,5.50,1.44,3.75,6.50
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.73,4.50,3.40,1.65,4.50,3.50,1.65,4.75,3.30,1.67


In [4]:
team_df = pd.read_sql("SELECT * FROM Team", connection)
team_df.head()

,id,team_api_id,team_fifa_api_id,team_long_name,team_short_name
0,1,9987,673.0,KRC Genk,GEN
1,2,9993,675.0,Beerschot AC,BAC
2,3,10000,15005.0,SV Zulte-Waregem,ZUL
3,4,9994,2007.0,Sporting Lokeren,LOK
4,5,9984,1750.0,KSV Cercle Brugge,CEB


In [5]:
team_attributes_df = pd.read_sql("SELECT * FROM Team_Attributes", connection)
team_attributes_df.head()

,id,team_fifa_api_id,team_api_id,date,buildUpPlaySpeed,buildUpPlaySpeedClass,buildUpPlayDribbling,buildUpPlayDribblingClass,buildUpPlayPassing,buildUpPlayPassingClass,...,chanceCreationShooting,chanceCreationShootingClass,chanceCreationPositioningClass,defencePressure,defencePressureClass,defenceAggression,defenceAggressionClass,defenceTeamWidth,defenceTeamWidthClass,defenceDefenderLineClass
0,1,434,9930,2010-02-22 00:00:00,60,Balanced,NaN,Little,50,Mixed,...,55,Normal,Organised,50,Medium,55,Press,45,Normal,Cover
1,2,434,9930,2014-09-19 00:00:00,52,Balanced,48.0,Normal,56,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
2,3,434,9930,2015-09-10 00:00:00,47,Balanced,41.0,Normal,54,Mixed,...,64,Normal,Organised,47,Medium,44,Press,54,Normal,Cover
3,4,77,8485,2010-02-22 00:00:00,70,Fast,NaN,Little,70,Long,...,70,Lots,Organised,60,Medium,70,Double,70,Wide,Cover
4,5,77,8485,2011-02-22 00:00:00,47,Balanced,NaN,Little,52,Mixed,...,52,Normal,Organised,47,Medium,47,Press,52,Normal,Cover


In [6]:
player_df = pd.read_sql("SELECT * FROM Player", connection)
player_df.head()

,id,player_api_id,player_name,player_fifa_api_id,birthday,height,weight
0,1,505942,Aaron Appindangoye,218353,1992-02-29 00:00:00,182.88,187
1,2,155782,Aaron Cresswell,189615,1989-12-15 00:00:00,170.18,146
2,3,162549,Aaron Doran,186170,1991-05-13 00:00:00,170.18,163
3,4,30572,Aaron Galindo,140161,1982-05-08 00:00:00,182.88,198
4,5,23780,Aaron Hughes,17725,1979-11-08 00:00:00,182.88,154


In [7]:
player_attributes_df = pd.read_sql("SELECT * FROM Player_Attributes", connection)
player_attributes_df.head()

,id,player_fifa_api_id,player_api_id,date,overall_rating,potential,preferred_foot,attacking_work_rate,defensive_work_rate,crossing,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,1,218353,505942,2016-02-18 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
1,2,218353,505942,2015-11-19 00:00:00,67.0,71.0,right,medium,medium,49.0,...,54.0,48.0,65.0,69.0,69.0,6.0,11.0,10.0,8.0,8.0
2,3,218353,505942,2015-09-21 00:00:00,62.0,66.0,right,medium,medium,49.0,...,54.0,48.0,65.0,66.0,69.0,6.0,11.0,10.0,8.0,8.0
3,4,218353,505942,2015-03-20 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0
4,5,218353,505942,2007-02-22 00:00:00,61.0,65.0,right,medium,medium,48.0,...,53.0,47.0,62.0,63.0,66.0,5.0,10.0,9.0,7.0,7.0


In [8]:
league_df = pd.read_sql("SELECT * FROM League", connection)
league_df.head()

,id,country_id,name
0,1,1,Belgium Jupiler League
1,1729,1729,England Premier League
2,4769,4769,France Ligue 1
3,7809,7809,Germany 1. Bundesliga
4,10257,10257,Italy Serie A


In [9]:
country_df = pd.read_sql("SELECT * FROM Country", connection)
country_df.head()

,id,name
0,1,Belgium
1,1729,England
2,4769,France
3,7809,Germany
4,10257,Italy


In [10]:
datasets = {
    "Match": match_df,
    "Team": team_df,
    "Team_Attributes": team_attributes_df,
    "Player": player_df,
    "Player_Attributes": player_attributes_df,
    "League": league_df,
    "Country": country_df
}

for name, df in datasets.items():
    print(f"{name}: {df.shape}")

Match: (25979, 115)
Team: (299, 5)
Team_Attributes: (1458, 25)
Player: (11060, 7)
Player_Attributes: (183978, 42)
League: (11, 3)
Country: (11, 2)


----

## Dataset V2.0 — Relational Data Integration

This section combines information from the original European Soccer Database, using the `Match` table as the foundation. The objective is to integrate relevant team, league, country, and historical attribute information while preserving one record per match and preventing data leakage.

### 1. Creating Home and Away Table Copies

Separate copies of the Team, Player, and Attribute tables are created with `home_` and `away_` prefixes. This allows us to distinguish between both sides during the joining process and avoid duplicate column names.

In [11]:
# Team
home_team_df = team_df.copy().add_prefix("home_")
away_team_df = team_df.copy().add_prefix("away_")

# Team Attributes
home_team_attributes_df = team_attributes_df.copy().add_prefix("home_")
away_team_attributes_df = team_attributes_df.copy().add_prefix("away_")

# Player
home_player_df = player_df.copy().add_prefix("home_")
away_player_df = player_df.copy().add_prefix("away_")

# Player Attributes
home_player_attributes_df = player_attributes_df.copy().add_prefix("home_")
away_player_attributes_df = player_attributes_df.copy().add_prefix("away_")

### 2. Joining Home and Away Team Information

The Home and Away Team tables are merged with the base `Match` dataset using their respective team API IDs. Left joins are used to preserve all matches while incorporating team information.

In [12]:
# Start with Match table
v2_df = match_df.copy()

# Join Home Team
v2_df = v2_df.merge(
    home_team_df.drop(columns=["home_id"]),
    on="home_team_api_id",
    how="left",
    validate="many_to_one"
)

# Join Away Team
v2_df = v2_df.merge(
    away_team_df.drop(columns=["away_id"]),
    on="away_team_api_id",
    how="left",
    validate="many_to_one"
)

print(v2_df.shape)
v2_df.head()

(25979, 121)


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,GBA,BSH,BSD,BSA,home_team_fifa_api_id,home_team_long_name,home_team_short_name,away_team_fifa_api_id,away_team_long_name,away_team_short_name
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,4.00,1.73,3.40,4.20,673.0,KRC Genk,GEN,675.0,Beerschot AC,BAC
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.75,1.91,3.25,3.60,15005.0,SV Zulte-Waregem,ZUL,2007.0,Sporting Lokeren,LOK
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,2.50,2.30,3.20,2.75,1750.0,KSV Cercle Brugge,CEB,229.0,RSC Anderlecht,AND
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,5.50,1.44,3.75,6.50,674.0,KAA Gent,GEN,1747.0,RAEC Mons,MON
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,1.65,4.75,3.30,1.67,NaN,FCV Dender EH,DEN,232.0,Standard de Liège,STL


### 3. Integrating League and Country Information

League and Country tables are merged using their corresponding identifiers. This provides additional information about the competition and country associated with each match.

In [13]:
# Rename columns
league_data = league_df[["id", "name"]].rename(
    columns={"id": "league_id", "name": "league_name"}
)

country_data = country_df.rename(
    columns={"id": "country_id", "name": "country_name"}
)

# Join League
v2_df = v2_df.merge(
    league_data,
    on="league_id",
    how="left",
    validate="many_to_one"
)

# Join Country
v2_df = v2_df.merge(
    country_data,
    on="country_id",
    how="left",
    validate="many_to_one"
)

print(v2_df.shape)
v2_df.head()

(25979, 123)


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,BSD,BSA,home_team_fifa_api_id,home_team_long_name,home_team_short_name,away_team_fifa_api_id,away_team_long_name,away_team_short_name,league_name,country_name
0,1,1,1,2008/2009,1,2008-08-17 00:00:00,492473,9987,9993,1,...,3.40,4.20,673.0,KRC Genk,GEN,675.0,Beerschot AC,BAC,Belgium Jupiler League,Belgium
1,2,1,1,2008/2009,1,2008-08-16 00:00:00,492474,10000,9994,0,...,3.25,3.60,15005.0,SV Zulte-Waregem,ZUL,2007.0,Sporting Lokeren,LOK,Belgium Jupiler League,Belgium
2,3,1,1,2008/2009,1,2008-08-16 00:00:00,492475,9984,8635,0,...,3.20,2.75,1750.0,KSV Cercle Brugge,CEB,229.0,RSC Anderlecht,AND,Belgium Jupiler League,Belgium
3,4,1,1,2008/2009,1,2008-08-17 00:00:00,492476,9991,9998,5,...,3.75,6.50,674.0,KAA Gent,GEN,1747.0,RAEC Mons,MON,Belgium Jupiler League,Belgium
4,5,1,1,2008/2009,1,2008-08-16 00:00:00,492477,7947,9985,1,...,3.30,1.67,NaN,FCV Dender EH,DEN,232.0,Standard de Liège,STL,Belgium Jupiler League,Belgium


### 4. Preparing Historical Team Attributes

Historical Team Attributes are prepared separately for Home and Away teams. Date columns are converted into datetime format to ensure that attributes can be matched accurately with the dates of individual matches.

In [14]:
# Convert match date
v2_df["date"] = pd.to_datetime(v2_df["date"])

# Home Team Attributes
home_attr = home_team_attributes_df.drop(
    columns=["home_id", "home_team_fifa_api_id"]
).rename(columns={"home_date": "home_attributes_date"})

# Away Team Attributes
away_attr = away_team_attributes_df.drop(
    columns=["away_id", "away_team_fifa_api_id"]
).rename(columns={"away_date": "away_attributes_date"})

# Convert attribute dates
home_attr["home_attributes_date"] = pd.to_datetime(
    home_attr["home_attributes_date"]
)

away_attr["away_attributes_date"] = pd.to_datetime(
    away_attr["away_attributes_date"]
)

### 5. Joining Historical Team Attributes

Historical team attributes are integrated using `merge_asof()`, selecting the most recent available attribute record before each match. This prevents future tactical information from being used in earlier matches and avoids duplicating match records.

In [15]:
# Home Team Attributes
v2_df = pd.merge_asof(
    v2_df.sort_values("date"),
    home_attr.sort_values("home_attributes_date"),
    left_on="date",
    right_on="home_attributes_date",
    by="home_team_api_id",
    direction="backward",
    allow_exact_matches=False
)

# Away Team Attributes
v2_df = pd.merge_asof(
    v2_df.sort_values("date"),
    away_attr.sort_values("away_attributes_date"),
    left_on="date",
    right_on="away_attributes_date",
    by="away_team_api_id",
    direction="backward",
    allow_exact_matches=False
)

# Restore original match order
v2_df = v2_df.sort_values("id").reset_index(drop=True)

print(v2_df.shape)

(25979, 167)


### 6. Validating Dataset Integrity

The combined dataset is checked for missing team information, unavailable historical attributes, duplicated matches, and changes in the total number of records. These checks help confirm that the joining process has preserved the original match structure.

In [16]:
print("Total matches:", len(v2_df))
print("Unique matches:", v2_df["id"].nunique())

print("Missing Home Team Names:", v2_df["home_team_long_name"].isna().sum())
print("Missing Away Team Names:", v2_df["away_team_long_name"].isna().sum())

print("Missing Home Attributes:", v2_df["home_attributes_date"].isna().sum())
print("Missing Away Attributes:", v2_df["away_attributes_date"].isna().sum())

assert len(v2_df) == len(match_df)
assert v2_df["id"].is_unique

Total matches: 25979
Unique matches: 25979
Missing Home Team Names: 0
Missing Away Team Names: 0
Missing Home Attributes: 6066
Missing Away Attributes: 6060


---

## Player and Historical Player Attribute Integration

This section integrates player information and historical FIFA attributes into Dataset V2.0. Each Home and Away player is linked to the Player table, followed by the most recent available attribute record before the corresponding match.

The objective is to incorporate individual player characteristics while preventing future-data leakage and preserving one record per match.

In [17]:
lineup_records = []

for side in ["home", "away"]:
    for number in range(1, 12):

        position = f"{side}_player_{number}"

        temp = v2_df[["id", "date", position]].copy()

        temp.columns = ["match_id", "date", "player_api_id"]
        temp["position"] = position

        lineup_records.append(temp)

player_lineups = pd.concat(lineup_records, ignore_index=True)

print("Total lineup positions:", len(player_lineups))
print("Missing player IDs:", player_lineups["player_api_id"].isna().sum())

player_lineups.head()

Total lineup positions: 571538
Missing player IDs: 29257


,match_id,date,player_api_id,position
0,1,2008-08-17,NaN,home_player_1
1,2,2008-08-16,NaN,home_player_1
2,3,2008-08-16,NaN,home_player_1
3,4,2008-08-17,NaN,home_player_1
4,5,2008-08-16,NaN,home_player_1


In [18]:
# Remove lineup positions without player IDs
player_lineups = player_lineups.dropna(
    subset=["player_api_id"]
).copy()

player_lineups["player_api_id"] = player_lineups[
    "player_api_id"
].astype("int64")

# Prepare Player table
player_info = player_df.drop(columns=["id"]).copy()

# Join Player information
player_lineups = player_lineups.merge(
    player_info,
    on="player_api_id",
    how="left",
    validate="many_to_one"
)

# Calculate player age at the time of the match
player_lineups["birthday"] = pd.to_datetime(
    player_lineups["birthday"]
)

player_lineups["age_at_match"] = (
    player_lineups["date"] - player_lineups["birthday"]
).dt.days / 365.25

print("Dataset shape:", player_lineups.shape)
print("Missing player names:", player_lineups["player_name"].isna().sum())

player_lineups.head()

Dataset shape: (542281, 10)
Missing player names: 0


,match_id,date,player_api_id,position,player_name,player_fifa_api_id,birthday,height,weight,age_at_match
0,145,2009-02-28,39890,home_player_1,Mark Volders,155841,1977-04-13,187.96,183,31.879535
1,146,2009-02-27,38327,home_player_1,Wouter Biebauw,151885,1984-05-21,187.96,181,24.772074
2,147,2009-02-28,95597,home_player_1,Bertrand Laquait,116334,1977-04-13,185.42,176,31.879535
3,149,2009-03-01,30934,home_player_1,Boubacar Barry Copa,120309,1979-12-30,180.34,152,29.169062
4,150,2009-02-28,37990,home_player_1,Stijn Stijnen,100033,1981-04-07,187.96,183,27.895962


In [19]:
# Prepare Player Attributes
player_attrs = player_attributes_df.copy()

player_attrs = player_attrs.drop(
    columns=["player_fifa_api_id"]
).rename(
    columns={
        "id": "attribute_record_id",
        "date": "attributes_date"
    }
)

# Convert dates
player_attrs["attributes_date"] = pd.to_datetime(
    player_attrs["attributes_date"]
)

# Join historical attributes
player_lineups = pd.merge_asof(
    player_lineups.sort_values("date"),
    player_attrs.sort_values(
        ["attributes_date", "attribute_record_id"]
    ),
    left_on="date",
    right_on="attributes_date",
    by="player_api_id",
    direction="backward",
    allow_exact_matches=False
)

print("Dataset shape:", player_lineups.shape)

print(
    "Missing historical ratings:",
    player_lineups["overall_rating"].isna().sum()
)

player_lineups.head()

Dataset shape: (542281, 50)
Missing historical ratings: 391


,match_id,date,player_api_id,position,player_name,player_fifa_api_id,birthday,height,weight,age_at_match,...,vision,penalties,marking,standing_tackle,sliding_tackle,gk_diving,gk_handling,gk_kicking,gk_positioning,gk_reflexes
0,4771,2008-08-09,46656,home_player_9,Jean-Michel Lesage,3189,1977-05-01,177.80,161,31.274470,...,73.0,70.0,47.0,33.0,25.0,14.0,9.0,13.0,15.0,13.0
1,19694,2008-08-09,23335,home_player_2,Jackie McNamara,4173,1973-10-24,175.26,143,34.792608,...,NaN,82.0,71.0,62.0,NaN,10.0,21.0,78.0,21.0,21.0
2,4769,2008-08-09,26279,away_player_3,Jean-Jacques Pierre,173483,1981-01-23,180.34,176,27.542779,...,50.0,64.0,65.0,74.0,56.0,9.0,23.0,46.0,23.0,23.0
3,4778,2008-08-09,40016,home_player_3,Milan Bisevac,178111,1983-08-31,185.42,187,24.941821,...,65.0,63.0,69.0,77.0,79.0,8.0,20.0,55.0,20.0,20.0
4,4777,2008-08-09,26345,home_player_3,Damien Perquis,135218,1984-04-10,185.42,165,24.331280,...,60.0,60.0,74.0,72.0,69.0,8.0,22.0,58.0,22.0,22.0


In [20]:
# Select player features
player_features = [
    "player_name",
    "birthday",
    "height",
    "weight",
    "age_at_match",
    "attributes_date"
]

player_features += [
    col for col in player_attrs.columns
    if col not in [
        "player_api_id",
        "attributes_date",
        "attribute_record_id"
    ]
]

# Restore one row per match
player_wide = player_lineups.set_index(
    ["match_id", "position"]
)[player_features].unstack("position")

# Rename columns
player_wide.columns = [
    f"{position}_{feature}"
    for feature, position in player_wide.columns
]

player_wide = player_wide.reset_index()

print("Player feature dataset:", player_wide.shape)

player_wide.head()

Player feature dataset: (25221, 969)


,match_id,away_player_1_player_name,away_player_10_player_name,away_player_11_player_name,away_player_2_player_name,away_player_3_player_name,away_player_4_player_name,away_player_5_player_name,away_player_6_player_name,away_player_7_player_name,...,home_player_10_gk_reflexes,home_player_11_gk_reflexes,home_player_2_gk_reflexes,home_player_3_gk_reflexes,home_player_4_gk_reflexes,home_player_5_gk_reflexes,home_player_6_gk_reflexes,home_player_7_gk_reflexes,home_player_8_gk_reflexes,home_player_9_gk_reflexes
0,145,Davy Schollen,Mbark Boussoufa,Matias Suarez,Olivier Deschacht,Arnold Kruiswijk,Marcin Wasilewski,Roland Juhasz,Thomas Chatelle,Jan Polak,...,NaN,21.0,NaN,20.0,20.0,22.0,NaN,NaN,NaN,22.0
1,146,Davino Verhulst,Stein Huysegems,Jelle Vossen,Joao Carlos,Dimitri Daeselaire,David Hubert,Anele Ngcongca,Tom de Mul,Daniel Pudil,...,20.0,20.0,20.0,20.0,22.0,20.0,21.0,23.0,25.0,22.0
2,147,Frederic Herpoel,Hocine Ragued,Moussa Gueye,Frederic Jay,Roberto Mirri,Francesco Migliore,Ivica Dzidic,Alesandro Cordaro,Cedric Collet,...,20.0,20.0,NaN,NaN,21.0,21.0,22.0,23.0,NaN,10.0
3,148,Bram Verbist,Dominic Foley,Stijn De Smet,Denis Viane,Anthony Portier,Dejan Kelhar,Arnar Vidarsson,Thomas Buffel,Sergiy Serebrennikov,...,20.0,20.0,20.0,21.0,20.0,20.0,23.0,20.0,22.0,20.0
4,149,Bojan Jorgacevic,Zlatan Ljubijankic,Mbaye Leye,Erlend Hanstveit,Jonas De Roeck,Stef Wils,NaN,Tim Smolders,Christophe Grondin,...,NaN,21.0,22.0,24.0,23.0,21.0,20.0,22.0,23.0,23.0


In [21]:
v2_df = v2_df.merge(
    player_wide,
    left_on="id",
    right_on="match_id",
    how="left",
    validate="one_to_one"
)

v2_df = v2_df.drop(columns=["match_id"])

print("Final dataset shape:", v2_df.shape)

v2_df.head()

Final dataset shape: (25979, 1135)


,id,country_id,league_id,season,stage,date,match_api_id,home_team_api_id,away_team_api_id,home_team_goal,...,home_player_10_gk_reflexes,home_player_11_gk_reflexes,home_player_2_gk_reflexes,home_player_3_gk_reflexes,home_player_4_gk_reflexes,home_player_5_gk_reflexes,home_player_6_gk_reflexes,home_player_7_gk_reflexes,home_player_8_gk_reflexes,home_player_9_gk_reflexes
0,1,1,1,2008/2009,1,2008-08-17,492473,9987,9993,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,2,1,1,2008/2009,1,2008-08-16,492474,10000,9994,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,3,1,1,2008/2009,1,2008-08-16,492475,9984,8635,0,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,4,1,1,2008/2009,1,2008-08-17,492476,9991,9998,5,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,5,1,1,2008/2009,1,2008-08-16,492477,7947,9985,1,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


In [24]:
print("Total matches:", len(v2_df))
print("Unique matches:", v2_df["id"].nunique())

print(
    "Missing Home Player 1 Names:",
    v2_df["home_player_1_player_name"].isna().sum()
)

print(
    "Missing Away Player 1 Names:",
    v2_df["away_player_1_player_name"].isna().sum()
)

print(
    "Missing Home Player 1 Ratings:",
    v2_df["home_player_1_overall_rating"].isna().sum()
)

print(
    "Missing Away Player 1 Ratings:",
    v2_df["away_player_1_overall_rating"].isna().sum())

print(f"The Data set size: {v2_df.shape}")

Total matches: 25979
Unique matches: 25979
Missing Home Player 1 Names: 1224
Missing Away Player 1 Names: 1234
Missing Home Player 1 Ratings: 1224
Missing Away Player 1 Ratings: 1234
The Data set size: (25979, 1135)


### Conclusion — Dataset V2.0 Integration

The integration process successfully combined match, team, league, country, historical team attributes, player information, and historical player attributes into a single dataset containing **25,979 unique matches and 1,135 columns**.

All original matches were preserved without duplication. However, missing player information and historical attributes remain and require further investigation.

**Next Step:** The dataset will undergo detailed data profiling, leakage identification, missing-value analysis, and feature preparation before model experimentation begins.

----

In [25]:
column_report = pd.DataFrame({
    "Column": v2_df.columns,
    "Data Type": v2_df.dtypes.astype(str).values,
    "Missing Values": v2_df.isna().sum().values,
    "Missing %": (v2_df.isna().mean() * 100).round(2).values,
    "Unique Values": v2_df.nunique(dropna=True).values
})

print("Total Columns:", len(column_report))

display(column_report.head(20))

Total Columns: 1135


,Column,Data Type,Missing Values,Missing %,Unique Values
0,id,int64,0,0.00,25979
1,country_id,int64,0,0.00,11
2,league_id,int64,0,0.00,11
3,season,str,0,0.00,8
4,stage,int64,0,0.00,38
5,date,datetime64[us],0,0.00,1694
6,match_api_id,int64,0,0.00,25979
7,home_team_api_id,int64,0,0.00,299
8,away_team_api_id,int64,0,0.00,299
9,home_team_goal,int64,0,0.00,11


In [26]:
display(
    column_report.sort_values(
        "Missing %", ascending=False
    ).head(30)
)

,Column,Data Type,Missing Values,Missing %,Unique Values
148,away_buildUpPlayDribbling,float64,20113,77.42,48
126,home_buildUpPlayDribbling,float64,20113,77.42,48
99,PSA,float64,14811,57.01,1475
97,PSH,float64,14811,57.01,948
98,PSD,float64,14811,57.01,665
112,BSH,float64,11818,45.49,101
109,GBH,float64,11817,45.49,159
111,GBA,float64,11817,45.49,172
113,BSD,float64,11818,45.49,59
114,BSA,float64,11818,45.49,96


In [27]:
display(
    column_report[column_report["Unique Values"] <= 1]
)

,Column,Data Type,Missing Values,Missing %,Unique Values
45,away_player_Y2,float64,1832,7.05,1


In [28]:
display(
    column_report[column_report["Unique Values"] == 25979]
)

,Column,Data Type,Missing Values,Missing %,Unique Values
0,id,int64,0,0.0,25979
6,match_api_id,int64,0,0.0,25979


In [29]:
v2_df.dtypes.value_counts()

float64           952
str               127
datetime64[us]     47
int64               9
Name: count, dtype: int64

In [31]:
print("No Missing Values:", (column_report["Missing %"] == 0).sum())

print("Below 10%:", (column_report["Missing %"] < 10).sum())

print("Above 50%:", (column_report["Missing %"] > 50).sum())

print("Completely Missing:", (column_report["Missing %"] == 100).sum())

No Missing Values: 17
Below 10%: 1053
Above 50%: 5
Completely Missing: 0


In [35]:
# Create target variable
v2_df["FTR"] = 0

v2_df.loc[
    v2_df["home_team_goal"] > v2_df["away_team_goal"],
    "FTR"
] = 1

v2_df.loc[
    v2_df["home_team_goal"] < v2_df["away_team_goal"],
    "FTR"
] = 2

# Verify target distribution
print(v2_df["FTR"].value_counts().sort_index())

FTR
0     6596
1    11917
2     7466
Name: count, dtype: int64


---

## Identifying and Removing Data Leakage

This section identifies features containing information generated during or after a match. Since the objective is to predict football outcomes before kickoff, these features must be excluded from the modelling dataset to prevent data leakage and ensure realistic model evaluation.

In [36]:
post_match_columns = [
    "home_team_goal",
    "away_team_goal",
    "goal",
    "shoton",
    "shotoff",
    "foulcommit",
    "card",
    "cross",
    "corner",
    "possession"
]

# Check which columns exist
leakage_columns = [
    col for col in post_match_columns
    if col in v2_df.columns
]

print("Post-match columns identified:", len(leakage_columns))
print(leakage_columns)

Post-match columns identified: 10
['home_team_goal', 'away_team_goal', 'goal', 'shoton', 'shotoff', 'foulcommit', 'card', 'cross', 'corner', 'possession']


In [37]:
# Create a separate working dataset
v2_clean_df = v2_df.drop(columns=leakage_columns).copy()

print("Original Dataset:", v2_df.shape)
print("Dataset After Removing Post-Match Columns:", v2_clean_df.shape)

Original Dataset: (25979, 1136)
Dataset After Removing Post-Match Columns: (25979, 1126)


In [38]:
identifier_columns = [
    "id",
    "match_api_id",
    "country_id",
    "league_id",
    "home_team_api_id",
    "away_team_api_id",
    "home_team_fifa_api_id",
    "away_team_fifa_api_id"
]

# Add all 22 player identifier columns
identifier_columns += [f"home_player_{i}" for i in range(1, 12)]
identifier_columns += [f"away_player_{i}" for i in range(1, 12)]

# Remove identifiers and create Dataset V2.0
data_v2 = v2_clean_df.drop(columns=identifier_columns).copy()

print("Dataset V2.0 Shape:", data_v2.shape)
data_v2.head()

Dataset V2.0 Shape: (25979, 1096)


,season,stage,date,home_player_X1,home_player_X2,home_player_X3,home_player_X4,home_player_X5,home_player_X6,home_player_X7,...,home_player_11_gk_reflexes,home_player_2_gk_reflexes,home_player_3_gk_reflexes,home_player_4_gk_reflexes,home_player_5_gk_reflexes,home_player_6_gk_reflexes,home_player_7_gk_reflexes,home_player_8_gk_reflexes,home_player_9_gk_reflexes,FTR
0,2008/2009,1,2008-08-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
1,2008/2009,1,2008-08-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,0
2,2008/2009,1,2008-08-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2
3,2008/2009,1,2008-08-17,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,1
4,2008/2009,1,2008-08-16,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,2


## Conclusion — Dataset V2.0 Construction

Dataset V2.0 was reconstructed from the original European Soccer Database by integrating all seven relational tables, including match information, team identities, league and country information, historical team attributes, player information, and historical player attributes.

The integration successfully preserved **25,979 unique matches without duplication**. Historical attributes were matched using records dated before each match to avoid introducing future information.

The target variable `FTR` was created to represent **Draw (0), Home Win (1), and Away Win (2)**. Ten post-match statistical columns and the main unnecessary identifiers were excluded, producing the working dataset `data_v2`.

**Next Phase:** Dataset V2.0 will undergo detailed data cleaning and preprocessing, including missing-value treatment, data-type corrections, invalid-value investigation, and further checks for redundant or potentially unavailable features before model experimentation.

---